# Semantic document search — prototype

This notebook prototypes the embedding approach used by the CRM platform's "smart search" feature (`/api/v1/search/semantic` in the backend).

It uses `ai_core.embeddings.encode()` directly — the same function the FastAPI backend calls in production (`backend/app/ai/embedding_client.py` re-exports it unchanged) — so there is no drift between what's prototyped here and what actually runs.

Goal: given a natural-language query with little or no keyword overlap with a document's title/description, can embedding similarity still surface the right document? That's the case Postgres full-text search alone can't handle.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from ai_core.embeddings import MODEL_NAME, encode

print("Model:", MODEL_NAME)

## Sample document set

Synthetic titles + descriptions representative of what lands in the `documents` table (only `title` + `description` are embedded — see `_embedding_text` in `backend/app/services/embedding_service.py` — file *contents* aren't indexed in this phase).

In [ ]:
documents = [
    {"id": "doc-1", "title": "Q3 Roadmap", "description": "Planning doc for the next quarter, billing dashboard and invoice migration"},
    {"id": "doc-2", "title": "Employee Handbook 2026", "description": "PTO policy, remote work guidelines, and code of conduct"},
    {"id": "doc-3", "title": "Vendor Security Questionnaire", "description": "Responses to our SOC2 auditor's annual security review"},
    {"id": "doc-4", "title": "Series B Pitch Deck", "description": "Fundraising deck covering traction, market size, and the ask"},
    {"id": "doc-5", "title": "Incident Postmortem — Checkout Outage", "description": "Root cause analysis for the payment processor outage on July 14"},
    {"id": "doc-6", "title": "Customer Onboarding Checklist", "description": "Steps the CS team runs through for every new enterprise account"},
]

corpus_text = [f"{d['title']}\n{d['description']}" for d in documents]
corpus_embeddings = encode(corpus_text)
corpus_embeddings.shape

## Query ranking

`encode()` returns L2-normalized vectors, so cosine similarity is just a dot product.

In [ ]:
def rank(query: str, top_k: int = 3):
    query_vec = encode([query])[0]
    scores = corpus_embeddings @ query_vec
    order = np.argsort(-scores)[:top_k]
    return [(documents[i]["title"], round(float(scores[i]), 3)) for i in order]


queries = [
    "what happened during the payment outage",       # no keyword overlap with doc-5's title
    "how much vacation time do we get",               # no keyword overlap with doc-2
    "steps for getting a new enterprise client set up", # paraphrase of doc-6
    "fundraising materials for investors",             # paraphrase of doc-4
]

for q in queries:
    print(f"\nQuery: {q!r}")
    for title, score in rank(q):
        print(f"  {score:.3f}  {title}")

Each query above shares few or no exact words with its best-matching document's title, yet the correct document ranks first — this is exactly the gap Postgres `websearch_to_tsquery` keyword search can't close, and why `/search/semantic` blends both (see `backend/app/api/v1/search.py`).

## Model choice

Comparing the production model (`all-MiniLM-L6-v2`, 384-dim, ~80MB) against a smaller alternative (`paraphrase-MiniLM-L3-v2`, 384-dim, ~60MB) on the same queries, to sanity-check the size/quality tradeoff before committing to one.

In [ ]:
from sentence_transformers import SentenceTransformer

candidate = SentenceTransformer("paraphrase-MiniLM-L3-v2")
candidate_corpus = candidate.encode(corpus_text, normalize_embeddings=True)

def rank_candidate(query: str, top_k: int = 1):
    query_vec = candidate.encode([query], normalize_embeddings=True)[0]
    scores = candidate_corpus @ query_vec
    order = np.argsort(-scores)[:top_k]
    return documents[order[0]]["title"], float(scores[order[0]])

print(f"{'query':<55} {'all-MiniLM-L6-v2':<35} paraphrase-MiniLM-L3-v2")
for q in queries:
    main_top = rank(q, top_k=1)[0]
    cand_top = rank_candidate(q)
    print(f"{q:<55} {main_top[0]:<25}{main_top[1]:<10.3f}{cand_top[0]:<25}{cand_top[1]:.3f}")

Both models rank the same top result on this small sample, but `all-MiniLM-L6-v2` is the more widely-used, better-benchmarked default for general semantic search (see the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard)) and stays fast enough for synchronous-ish use in a `BackgroundTask` on CPU. That's the one wired into `ai_core/embeddings.py` and used by the backend.